In [1]:
#%%capture
#!pip install unsloth
# Also get the latest nightly Unsloth!
#!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [2]:
from unsloth import FastLanguageModel
import torch
import os
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "./local-DeepSeek-R1-Distill-Llama-8B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit
)

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.11.1: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 119.697 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0a0+50eac811a6.nv25.09. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [3]:
prompt_style = """Below is an instruction that describes a task, paired with an input that provides further context. 
Write a response that appropriately completes the request. 
Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.

### Instruction:
You are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning. 
Please answer the following medical question. 

### Question:
{}

### Response:
<think>{}"""

In [4]:
question = "A 61-year-old woman with a long history of involuntary urine loss during activities like coughing or sneezing but no leakage at night undergoes a gynecological exam and Q-tip test. Based on these findings, what are the possible causes for his symptoms could be?"


FastLanguageModel.for_inference(model) 
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=1200,
    use_cache=True,
)
response = tokenizer.batch_decode(outputs)
print(response[0].split("### Response:")[1])


<think>
Alright, so I have this 61-year-old woman who's been dealing with involuntary urine loss whenever she coughs or sneezes. She doesn't leak at night, though. She's had a gynecological exam and a Q-tip test. I need to figure out possible causes for her symptoms.

First, I know that involuntary urine loss during activities like coughing or sneezing is often related to the lower urinary tract. The bladder or urethra might not be closing properly after those activities. Since she doesn't leak at night, it's less likely to be something like nighttime urinary incontinence, which is more common in conditions like sleep apnea or neurogenerative disorders.

The gynecological exam probably checked for things like atrophy, infections, or structural issues in the genital area. The Q-tip test is used to assess urethral mobility and function. If the Q-tip stays in the correct position after movement, it might indicate that the urethra isn't closing properly, which could be contributing to the

In [5]:
train_prompt_style = """Below is an instruction that describes a task, paired with an input that provides further context. 
Write a response that appropriately completes the request. 
Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.

### Instruction:
You are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning. 
Please answer the following medical question. 

### Question:
{}

### Response:
<think>
{}
</think>
{}"""

In [6]:
EOS_TOKEN = tokenizer.eos_token  # Must add EOS_TOKEN

#CoT - Complex Chain of Thought
def formatting_prompts_func(examples):
    inputs = examples["Question"]
    cots = examples["Complex_CoT"]
    outputs = examples["Response"]
    texts = []
    for input, cot, output in zip(inputs, cots, outputs):
        text = train_prompt_style.format(input, cot, output) + EOS_TOKEN
        texts.append(text)
    return {
        "text": texts,
    }

In [8]:
from datasets import load_dataset, load_from_disk

# Load dataset from disk
dataset_on_disk = load_from_disk("./medical-o1-reasoning-SFT/hf_format", "en")
dataset_on_disk = dataset_on_disk.shuffle(seed=42)
#train_dataset = dataset_on_disk.select(range(16000))

print("Rows in dataset_on_disk:", len(dataset_on_disk))

# Apply formatting (this does NOT change row count, only columns)
dataset = dataset_on_disk.map(formatting_prompts_func,batched=True,)

print("Rows after formatting:", len(dataset))

# Inspect first row
print(dataset[0])


Rows in dataset_on_disk: 19704


Map:   0%|          | 0/19704 [00:00<?, ? examples/s]

Rows after formatting: 19704
{'Question': 'In the instrument formula for a Gingival Margin Trimmer (GMT) used during cavity preparation, what is the second number representing the angle of the cutting edge when access to the distal gingival margin is achieved?', 'Complex_CoT': "Alright, so a Gingival Margin Trimmer, or GMT for short, is some sort of dental tool used during cavity prep. I need to figure out what that second number in its formula really means, especially when working with the distal gingival margin. Let's start with the basics about these numbers.\n\nThe first number in any dental instrument formula usually tells us the blade width, measured in tenths of a millimeter. Pretty straightforward.\n\nNow, the second number is where it gets interesting. It refers to the angle of the blade relative to the handle's long axis. So, it sort of dictates how the blade sits or tilts compared to the main handle.\n\nOh yeah, then there’s the third number, which gives us the idea about th

In [9]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,  
    bias="none",  
    use_gradient_checkpointing="unsloth",  # True or "unsloth" for very long context
    random_state=3407,
    use_rslora=False,  
    loftq_config=None,
)

Unsloth 2025.11.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [10]:
# #DGX Spark specific

# from trl import SFTTrainer
# from transformers import TrainingArguments
# from unsloth import is_bfloat16_supported

# trainer = SFTTrainer(
#     model=model,
#     tokenizer=tokenizer,
#     train_dataset=dataset,
#     dataset_text_field="text",
#     max_seq_length=max_seq_length,
#     dataset_num_proc=4,
#     args=TrainingArguments(
#         per_device_train_batch_size=4,
#         gradient_accumulation_steps=8,
#         num_train_epochs=1,
#         warmup_ratio=0.03,
#         learning_rate=1e-4,
#         fp16=not is_bfloat16_supported(),
#         bf16=is_bfloat16_supported(),
#         logging_steps=25,
#         optim="adamw_8bit",
#         weight_decay=0.01,
#         lr_scheduler_type="cosine",
#         seed=3407,
#         output_dir="outputs",
#         save_steps=500,
#         save_total_limit=2,
#         report_to="none",
#     ),
# )


In [11]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="train",
    max_seq_length=max_seq_length,
    dataset_num_proc=4,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        # Use num_train_epochs = 1, warmup_ratio for full training runs!
        warmup_steps=5,
        max_steps=10,    #### modify as needed
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=24):   0%|          | 0/19704 [00:00<?, ? examples/s]

In [12]:
trainer_stats = trainer.train()


The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 19,704 | Num Epochs = 1 | Total steps = 10
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


* Trackio project initialized: huggingface
* Trackio metrics will be synced to Hugging Face Dataset: prabirmclean/trackio-dataset
* Found existing space: https://huggingface.co/spaces/prabirmclean/trackio
* View dashboard by going to: https://prabirmclean-trackio.hf.space/


* Created new run: prabirmclean-1770403452
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,1.950900


* Run finished. Uploading logs to Trackio (please wait...)


In [13]:
question = "A 61-year-old woman with a long history of involuntary urine loss during activities like coughing or sneezing but no leakage at night undergoes a gynecological exam and Q-tip test. Based on these findings, what would cystometry most likely reveal about her residual volume and detrusor contractions?"


FastLanguageModel.for_inference(model)  # Unsloth has 2x faster inference!
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=1200,
    use_cache=True,
)
response = tokenizer.batch_decode(outputs)
print(response[0].split("### Response:")[1])



<think>
Okay, so I'm trying to figure out what cystometry would show for this woman. She's 61 and has been dealing with involuntary urine loss whenever she coughs or sneezes, but she doesn't leak at night. She had a gynecological exam and a Q-tip test. I need to think about how these things might connect.

First, involuntary urine loss during activities like coughing or sneezing makes me think of something called "urgency" incontinence. It's usually due to an overactive bladder or maybe something like irritable bowel syndrome, which can sometimes affect the bladder. But it's also possible that she has urethritis, which is an inflammation of the urethra, or maybe even interstitial cystitis, a condition that causes pain and frequent urination.

Now, the Q-tip test: this is a diagnostic tool used to check for urethral obstruction. The idea is to insert a catheter (Q-tip) into the urethra and measure how far it can be pushed back into the bladder. If the obstruction is significant, it mig

In [14]:
new_model_local = "DeepSeek-R1-Medical-FT-8b-16bts"
model.save_pretrained(new_model_local) 
tokenizer.save_pretrained(new_model_local)

model.save_pretrained_merged(new_model_local, tokenizer, save_method = "merged_16bit",)

Detected local model directory: /projects/dgx-book/CH11-Finetuning/local-DeepSeek-R1-Distill-Llama-8B
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Unsloth: Preparing safetensor model files:  25%|████████████████████████████████████████████▌                                                                                                                                     | 1/4 [00:00<00:01,  2.76it/s]

Copied model-00004-of-00004.safetensors from local model directory


Unsloth: Preparing safetensor model files:  50%|█████████████████████████████████████████████████████████████████████████████████████████                                                                                         | 2/4 [00:01<00:01,  1.05it/s]

Copied model-00003-of-00004.safetensors from local model directory


...

Copied model-00001-of-00004.safetensors from local model directory


Unsloth: Preparing safetensor model files: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:04<00:00,  1.14s/it]


Copied model-00002-of-00004.safetensors from local model directory


Unsloth: Merging weights into 16bit: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [01:44<00:00, 26.14s/it]


Unsloth: Merge process complete. Saved to `/projects/dgx-book/CH11-Finetuning/DeepSeek-R1-Medical-FT-8b-16bts`
